### Create Bronze storage

In [0]:
from pyspark.sql import functions as F

# Store raw ingested transactions in a dedicated schema.
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.sentinel_bronze
""")

# Keep ingestion state separate from incoming transaction files.
spark.sql("""
    CREATE VOLUME IF NOT EXISTS
    workspace.sentinel_bronze.ingestion_state
""")

landing_path = (
    "/Volumes/workspace/sentinel_sources/replay_landing"
)

state_path = (
    "/Volumes/workspace/sentinel_bronze/"
    "ingestion_state/transactions_v1"
)

bronze_table = "workspace.sentinel_bronze.transactions"

### Ingest released files

In [0]:
# Auto Loader discovers Parquet files in landing.
# The partition folder supplies replay_batch.
# schemaLocation preserves the discovered source schema.
incoming = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{state_path}/schema")
    .option("cloudFiles.partitionColumns", "replay_batch")
    .load(landing_path)
)

# Preserve source fields and add ingestion traceability.
# This path identifies the released Parquet file;
# source_file still identifies the original CSV.
bronze = incoming.select(
    "*",
    F.col("_metadata.file_path").alias("_ingested_file_path"),
    F.current_timestamp().alias("_ingestion_timestamp")
)

# Append to a managed Delta table.
# The checkpoint records progress across executions.
# AvailableNow processes available files and then stops.
query = (
    bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{state_path}/checkpoint")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

print("Bronze ingestion completed.")

### Verify the first ingestion


In [0]:
# Read the persisted Bronze table.
saved_bronze = spark.table(bronze_table)

assert saved_bronze.count() == 2414, (
    "Expected 2,414 Bronze transactions"
)

assert saved_bronze.select("trans_num").distinct().count() == 2414, (
    "Expected 2,414 distinct transaction IDs"
)

assert "is_fraud" not in saved_bronze.columns, (
    "Fraud labels must not appear in Bronze"
)

display(
    saved_bronze
    .groupBy("replay_batch")
    .count()
    .orderBy("replay_batch")
)

print("Verified: 2,414 Bronze transactions with unique IDs.")

## Bronze ingestion verification

- Released batch: January 1, 2019.
- Expected and actual Bronze count: 2,414 transactions.
- Distinct transaction IDs: 2,414.
- Fraud labels excluded from Bronze.
- Ingestion file path and timestamp recorded.
- Reran ingestion using the same checkpoint.
- Count remained 2,414: no additional rows or duplicate IDs.
- Remaining batches are staged; evaluation batches remain unreleased.